# 03 · Pemrosesan: relevansi, peta vendor, overlap

**Analisis awal, bukan solusi final.** Notebook ini memakai tabel bersih dari `02` dan
menghitung jawaban sementara untuk alur yang direncanakan:

1. Profil Wooden Ships sebagai pembanding
2. Relevansi tiap produk (knitwear, kategori, bahan, harga)
3. Relevansi tiap brand, digabung dari semua toko
4. Reverse lookup: brand dijual di toko mana, dan overlap dengan pelanggan kita
5. Brand yang sering dijual bersama
6. Daftar toko peluang dan celah per wilayah
7. Contoh daftar resmi brand (bila berkasnya ada)

Semua ambang di bawah adalah **usulan** yang mudah diganti, bukan keputusan.

In [ ]:
import json

import nbutil as nb
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from scrapebot.analysis.inputs import core_name
from scrapebot.analysis.location import EARTH_MI

print("Run:", nb.run_id())
brand_stores = nb.load("brand_stores")
brands_dim = nb.load("brands_dim")
stores = nb.load("stores_clean")
knit = nb.load("knit_clean")
knit_text = nb.load("knit_text")
stockists = nb.load_stockists()

## Parameter

| Parameter | Nilai | Catatan |
|---|---|---|
| `PRICE_BAND` | 0,75 sampai 1,35 kali harga kita | Aturan yang sekarang dipakai sistem. Rentang "50–60%" belum diputuskan artinya (PRD, Q-10) |
| `MIN_STORES`, `MIN_PRODUCTS` | 8 toko, 30 produk | Di bawah ini sebuah brand tidak diberi peringkat |
| `RIVAL_KNIT_SHARE` | 25% | Batas antara pesaing knitwear dan brand penanda |
| `WATCH` | empat brand dari proposal | Diganti isi `brands.csv` bila berkasnya ada |

In [ ]:
WS = "woodenships"
PRICE_BAND = (0.75, 1.35)
MIN_STORES, MIN_PRODUCTS = 8, 30
RIVAL_KNIT_SHARE = 0.25
WATCH = ["whitewarren", "lillap", "rails", "zsupply"]

brands_file = nb.INPUTS / "brands.csv"
if brands_file.exists():
    listed = pd.read_csv(brands_file)["brand"].map(nb.compact).tolist()
    WATCH = [key for key in listed if key] or WATCH
name_of = brands_dim.set_index("brand_key")["brand"].to_dict()
print("Brand yang dipantau:", ", ".join(name_of.get(key, key) for key in WATCH))

## 1. Profil Wooden Ships

Idealnya profil ini datang dari katalog Shopify kita sendiri dan `price_points.csv`.
Selama keduanya belum ada, profil diambil dari produk Wooden Ships yang terlihat di katalog
stockist dalam run ini. Sampelnya kecil, jadi anggap ini pengganti sementara.

In [ ]:
ws_knit = knit[knit["brand_key"] == WS]
ws_categories = set(ws_knit["category"].dropna())
ws_families = set(ws_knit["material_family"].dropna())

prices_file = nb.INPUTS / "price_points.csv"
if prices_file.exists():
    ws_price = pd.read_csv(prices_file).set_index("category")["retail_usd"].dropna().to_dict()
    price_source = "price_points.csv"
else:
    ws_price = ws_knit.groupby("category")["price_usd"].median().dropna().to_dict()
    price_source = (
        f"{int(ws_knit['price_usd'].notna().sum())} produk Wooden Ships di katalog stockist"
    )
ws_price_all = float(np.median(list(ws_price.values()))) if ws_price else float("nan")

print(
    f"Produk knitwear Wooden Ships yang terlihat: {len(ws_knit)} di {ws_knit['domain'].nunique()} toko"
)
print("Kategori:", sorted(ws_categories))
print("Keluarga bahan:", sorted(ws_families))
print(f"Harga acuan per kategori (sumber: {price_source}):")
print(pd.Series(ws_price).round(0).to_string() if ws_price else "  (tidak ada)")

## 2. Relevansi produk

Knitwear adalah syarat masuk (tabel ini hanya berisi knitwear). Tiga ukuran lain punya tiga
kemungkinan nilai: ya, tidak, atau **tidak diketahui**. Produk disebut relevan bila semua
ukuran yang diketahui bernilai ya, dan minimal dua ukuran diketahui.

In [ ]:
def tri(flag: pd.Series, known: pd.Series) -> pd.Series:
    """True/False where the value is known, <NA> where it is not."""
    return flag.astype("boolean").where(known)


others = knit[knit["brand_key"] != WS].copy()
reference = others["category"].map(ws_price).fillna(ws_price_all)
others["price_ratio"] = (others["price_usd"] / reference).round(2)
others["cat_similar"] = tri(others["category"].isin(ws_categories), others["category"].notna())
others["mat_similar"] = tri(
    others["material_family"].isin(ws_families), others["material_family"].notna()
)
others["price_in_band"] = tri(
    others["price_ratio"].between(*PRICE_BAND), others["price_ratio"].notna()
)

parts = others[["cat_similar", "mat_similar", "price_in_band"]]
others["parts_known"] = parts.notna().sum(axis=1)
others["parts_yes"] = parts.fillna(False).sum(axis=1)
others["relevant"] = (others["parts_known"] >= 2) & (others["parts_yes"] == others["parts_known"])

summary = pd.DataFrame(
    {
        "ya": parts.apply(lambda s: int(s.fillna(False).sum())),
        "tidak": parts.apply(lambda s: int((~s).fillna(False).sum())),
        "tidak diketahui": parts.isna().sum(),
    }
)
summary.index = ["kategori serupa", "bahan serupa", "harga dalam rentang"]
print(
    f"Produk knitwear brand lain: {len(others):,} · relevan: {int(others['relevant'].sum()):,} ({others['relevant'].mean():.0%})"
)
summary

## 3. Relevansi brand

Sebuah brand dinilai dari **semua** toko yang menjualnya, dengan lima komponen yang
disimpan terpisah:

| Komponen | Arti |
|---|---|
| `knit_share` | Porsi produk brand itu yang knitwear |
| `stores` | Di berapa toko brand itu terlihat |
| `customer_share` | Porsi toko brand itu yang sudah pelanggan kita |
| `price_position` | Median harga knitwear-nya dibagi harga acuan kita (1,0 berarti setara) |
| `text_similarity` | Kemiripan teks produk (judul, tipe, tag) dengan produk Wooden Ships, TF-IDF |

`rank_score` adalah rata-rata peringkat persentil kelima komponen dengan bobot sama. Ini
peringkat sementara: bobot yang benar baru bisa ditentukan dengan daftar uji dari tim
wholesale.

In [ ]:
customers = set(stores.loc[stores["is_customer"], "domain"])
territory = stores.set_index("domain")["territory"].to_dict()
brand_stores["is_customer"] = brand_stores["domain"].isin(customers)
brand_stores["territory"] = brand_stores["domain"].map(territory)

brands = brand_stores.groupby("brand_key").agg(
    stores=("domain", "nunique"),
    products=("products", "sum"),
    knit_products=("knit_products", "sum"),
    customers=("is_customer", "sum"),
)
brands["brand"] = brands.index.map(name_of)
brands["knit_share"] = (brands["knit_products"] / brands["products"]).round(3)
brands["customer_share"] = (brands["customers"] / brands["stores"]).round(3)
by_brand = others.groupby("brand_key")
brands["price_position"] = by_brand["price_ratio"].median().round(2)
brands["relevant_share"] = by_brand["relevant"].mean().round(3)
brands["enough_data"] = (brands["stores"] >= MIN_STORES) & (brands["products"] >= MIN_PRODUCTS)
brands["kind"] = np.where(
    brands["knit_share"] >= RIVAL_KNIT_SHARE, "pesaing knitwear", "brand penanda"
)
brands["watched"] = brands.index.isin(WATCH)
print(f"Brand dengan cukup data: {int(brands['enough_data'].sum())} dari {len(brands):,}")

In [ ]:
# Text similarity: one document per brand from its knitwear titles, types and tags.
docs = (
    knit_text["title"].fillna("")
    + " "
    + knit_text["product_type"].fillna("")
    + " "
    + knit_text["tags"].fillna("")
).str.lower()
by_key = docs.groupby(knit_text["brand_key"]).agg(" ".join)
keep = [
    key
    for key in brands.index[brands["enough_data"] | brands["watched"]]
    if key in by_key.index and key != WS
]
if WS in by_key.index and keep:
    matrix = TfidfVectorizer(
        stop_words="english", ngram_range=(1, 2), min_df=2, sublinear_tf=True
    ).fit_transform(by_key.loc[[WS, *keep]])
    brands.loc[keep, "text_similarity"] = cosine_similarity(matrix[0], matrix[1:]).ravel().round(3)
else:
    brands["text_similarity"] = np.nan

ranked = brands[(brands["enough_data"] | brands["watched"]) & (brands.index != WS)].copy()
closeness = -np.log(ranked["price_position"].where(ranked["price_position"] > 0)).abs()
components = pd.DataFrame(
    {
        "knit_share": ranked["knit_share"],
        "stores": ranked["stores"],
        "customer_share": ranked["customer_share"],
        "price": closeness,
        "text": ranked["text_similarity"],
    }
)
ranked["rank_score"] = components.rank(pct=True).mean(axis=1).round(3)
ranked = ranked.sort_values("rank_score", ascending=False).reset_index()
columns = [
    "brand",
    "kind",
    "stores",
    "knit_share",
    "customer_share",
    "price_position",
    "text_similarity",
    "relevant_share",
    "rank_score",
    "watched",
]
ranked[columns].head(20)

In [ ]:
print("Brand yang dipantau:")
ranked.loc[ranked["watched"], columns]

## 4. Reverse lookup dan overlap

Peta brand dan toko dibalik: untuk tiap brand, di toko mana ia terlihat, berapa di antaranya
sudah pelanggan kita, dan berapa yang berada di wilayah tanpa stockist.

Semua angka di sini adalah **sampel**: hanya toko yang kebetulan terbaca.

In [ ]:
def overlap_of(keys: list[str]) -> pd.DataFrame:
    rows = brand_stores[brand_stores["brand_key"].isin(keys)]
    table = rows.groupby("brand_key").agg(
        stores=("domain", "nunique"),
        shared=("is_customer", "sum"),
        open_territory=(
            "territory",
            lambda s: int(((s == "wilayah kosong") & ~rows.loc[s.index, "is_customer"]).sum()),
        ),
        near_stockist=(
            "territory",
            lambda s: int(
                ((s == "ada stockist dalam radius") & ~rows.loc[s.index, "is_customer"]).sum()
            ),
        ),
    )
    table["not_ours"] = table["stores"] - table["shared"]
    table["location_unknown"] = table["not_ours"] - table["open_territory"] - table["near_stockist"]
    table.insert(0, "brand", table.index.map(name_of))
    table["coverage"] = "sampel"
    return table.sort_values("stores", ascending=False).reset_index()


brand_overlap = overlap_of(ranked["brand_key"].tolist())
brand_overlap[brand_overlap["brand_key"].isin(WATCH)]

In [ ]:
watched_rows = brand_stores[brand_stores["brand_key"].isin(WATCH)]
per_store = watched_rows.groupby("domain")["brand_key"].nunique()
info = stores.set_index("domain").loc[per_store.index]
print(f"Toko yang menjual minimal satu brand yang dipantau: {len(per_store)}")
print(f"  sudah pelanggan kita:        {int(info['is_customer'].sum())}")
print(
    f"  kandidat partner, bukan kita: {int(((info['segment'] == 'retail_partner') & ~info['is_customer']).sum())}"
)
print(f"  menjual dua brand atau lebih: {int((per_store >= 2).sum())}")
print(f"  menjual tiga brand atau lebih: {int((per_store >= 3).sum())}")

## 5. Brand yang sering dijual bersama

Dua ukuran:

- **Jaccard**: porsi toko yang sama di antara semua toko kedua brand. Dipakai untuk melihat
  brand mana yang distribusinya paling mirip.
- **Lift**: seberapa sering dua brand muncul bersama dibanding kalau kebetulan saja. Lift di
  atas 1 berarti keduanya memang cenderung dijual di toko yang sama.

In [ ]:
enough = ranked.loc[ranked["enough_data"], "brand_key"].tolist()
incidence = (
    pd.crosstab(brand_stores["domain"], brand_stores["brand_key"])
    .reindex(columns=enough, fill_value=0)
    .gt(0)
)
incidence[WS] = incidence.index.isin(customers)
counts = incidence.sum()
both = incidence.astype(int).T.dot(incidence.astype(int))
union = counts.to_numpy()[:, None] + counts.to_numpy()[None, :] - both.to_numpy()
jaccard = pd.DataFrame(
    np.where(union > 0, both.to_numpy() / np.maximum(union, 1), 0.0),
    index=both.index,
    columns=both.columns,
)

n_stores = len(incidence)
pairs = [
    {
        "a": name_of.get(a, "Wooden Ships"), "b": name_of.get(b, "Wooden Ships"),
        "a_key": a, "b_key": b, "both": int(both.loc[a, b]),
        "jaccard": round(float(jaccard.loc[a, b]), 3),
        "lift": round(float(both.loc[a, b] * n_stores / (counts[a] * counts[b])), 2),
    }
    for i, a in enumerate(both.index)
    for b in both.index[i + 1 :]
    if both.loc[a, b] >= 5
]  # fmt: skip
pair_table = pd.DataFrame(pairs).sort_values("lift", ascending=False).reset_index(drop=True)
print(
    f"{n_stores} toko · {len(pair_table)} pasangan brand yang muncul bersama di 5 toko atau lebih"
)
pair_table[["a", "b", "both", "jaccard", "lift"]].head(15)

In [ ]:
with_ws = (
    jaccard[WS]
    .drop(WS)
    .sort_values(ascending=False)
    .head(10)
    .round(3)
    .rename("jaccard dengan pelanggan kita")
)
print(
    f"Catatan: hanya {int(counts[WS])} pelanggan kita ada di antara toko terbaca, jadi angka ini sangat kasar."
)
with_ws.rename(index=name_of).to_frame()

## 6. Toko peluang

Toko yang terbaca, dinilai sebagai kandidat partner, dan belum pelanggan kita. Diurutkan
menurut jumlah brand relevan yang dijualnya, lalu skor dari tahap analisis.

"Brand relevan" di sini: pesaing knitwear dengan cukup data, ditambah brand yang dipantau.

In [ ]:
relevant_keys = set(
    ranked.loc[ranked["enough_data"] & (ranked["kind"] == "pesaing knitwear"), "brand_key"]
) | set(WATCH)
carried = (
    brand_stores[brand_stores["brand_key"].isin(relevant_keys)]
    .groupby("domain")["brand"]
    .agg(lambda s: sorted(set(s)))
    .rename("relevant_brands")
)
watched_count = watched_rows.groupby("domain")["brand_key"].nunique().rename("watched_brands")

candidates = stores[
    stores["readable"] & ~stores["is_customer"] & (stores["segment"] == "retail_partner")
]
opportunities = candidates.join(carried, on="domain").join(watched_count, on="domain")
opportunities["relevant_brands"] = opportunities["relevant_brands"].map(
    lambda v: list(v) if isinstance(v, (list, np.ndarray)) else []
)
opportunities["n_relevant"] = opportunities["relevant_brands"].map(len)
opportunities["watched_brands"] = opportunities["watched_brands"].fillna(0).astype(int)
opportunities = opportunities.sort_values(["n_relevant", "score"], ascending=False).reset_index(
    drop=True
)

print(f"Kandidat partner yang belum pelanggan: {len(opportunities)}")
print(f"  menjual minimal satu brand relevan:  {int((opportunities['n_relevant'] > 0).sum())}")
print(f"  punya email:                         {int(opportunities['has_email'].sum())}")
print("\nMenurut teritori:")
print(opportunities["territory"].value_counts().to_string())
opportunities[
    ["domain", "city", "state", "territory", "has_email", "score", "n_relevant", "watched_brands"]
].head(15)

## 7. Celah per wilayah

Per negara bagian: jumlah toko peluang dibanding jumlah stockist kita. Hanya toko yang
lokasinya diketahui yang bisa dihitung.

In [ ]:
by_state = (
    opportunities[opportunities["state"].notna()]
    .groupby("state")
    .agg(
        opportunities=("domain", "size"),
        open_territory=("territory", lambda s: int((s == "wilayah kosong").sum())),
    )
)
by_state["stockists"] = (
    stockists["prov_state"].str.upper().value_counts().reindex(by_state.index).fillna(0).astype(int)
)
regions = by_state.sort_values("opportunities", ascending=False).reset_index()
print(
    f"Toko peluang dengan lokasi: {int(regions['opportunities'].sum())} dari {len(opportunities)}"
)
regions.head(12)

## 8. Contoh daftar resmi brand

Bila ada berkas daftar retailer di `data/locators/` (format Stockist.co), tiap tokonya
dicocokkan dengan stockist kita: nama yang sama **dan** jarak di bawah 1 mil. Nama dan
negara bagian yang sama tapi lokasinya berbeda masuk daftar periksa, bukan dianggap cocok.

In [ ]:
def same_name(key: str, other: str) -> bool:
    return bool(key) and (other == key or (len(key) >= 5 and (key in other or other in key)))


def match_locator(rows: list[dict], brand: str) -> pd.DataFrame:
    points = stockists.dropna(subset=["lat", "lng"]).assign(key=lambda f: f["name"].map(core_name))
    lat2, lng2 = np.radians(points["lat"].to_numpy()), np.radians(points["lng"].to_numpy())
    keys, regions_ = points["key"].tolist(), points["prov_state"].str.upper().tolist()
    out = []
    for row in rows:
        key, state = core_name(row["name"]), str(row.get("state") or "").upper()
        lat1, lng1 = np.radians(float(row["latitude"])), np.radians(float(row["longitude"]))
        a = (
            np.sin((lat2 - lat1) / 2) ** 2
            + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2
        )
        distance = 2 * EARTH_MI * np.arcsin(np.sqrt(np.minimum(1.0, a)))
        if any(d <= 1.0 and same_name(key, other) for d, other in zip(distance, keys, strict=True)):
            group = "toko yang sama"
        elif any(
            key and other == key and region == state
            for other, region in zip(keys, regions_, strict=True)
        ):
            group = "perlu diperiksa"
        elif distance.min() <= 15:
            group = "bukan kita, ada stockist dalam 15 mil"
        else:
            group = "bukan kita, wilayah kosong"
        out.append(
            {
                "brand": brand,
                "name": row["name"],
                "city": row.get("city"),
                "state": row.get("state"),
                "group": group,
                "nearest_stockist_miles": round(float(distance.min()), 1),
            }
        )
    return pd.DataFrame(out)


files = sorted(nb.LOCATORS.glob("*.json")) if nb.LOCATORS.exists() else []
if files:
    locator = pd.concat(
        [match_locator(json.loads(f.read_text(encoding="utf-8")), f.stem.title()) for f in files],
        ignore_index=True,
    )
    nb.save(locator, "locator_match")
    print(pd.crosstab(locator["group"], locator["brand"]).to_string())
else:
    locator = pd.DataFrame()
    print("Tidak ada berkas di data/locators/, bagian ini dilewati.")

In [ ]:
if len(locator):
    states = locator.groupby(["brand", "state"]).size().rename("brand_retailers").reset_index()
    states["stockists"] = (
        states["state"]
        .map(stockists["prov_state"].str.upper().value_counts())
        .fillna(0)
        .astype(int)
    )
    states["gap"] = states["brand_retailers"] - states["stockists"]
    nb.save(states, "locator_states")
    display(states.sort_values("brand_retailers", ascending=False).head(10))

## 9. Simpan hasil

In [ ]:
relevance_columns = [
    "domain",
    "brand_key",
    "title",
    "category",
    "material_family",
    "price_usd",
    "price_ratio",
    "cat_similar",
    "mat_similar",
    "price_in_band",
    "relevant",
]
nb.save(others[relevance_columns], "product_relevance")
nb.save(ranked, "brands_ranked")
nb.save(brand_overlap, "brand_overlap")
nb.save(jaccard.reset_index().rename(columns={"brand_key": "key"}), "brand_jaccard")
nb.save(pair_table, "brand_pairs")
nb.save(opportunities, "opportunities")
nb.save(regions, "regions")
print("Disimpan di:", nb.out_dir().relative_to(nb.ROOT))

## Batasan hasil ini

- **Sampel, bukan sensus.** Sebuah brand hanya terlihat di toko yang terbaca.
- **Profil Wooden Ships masih pengganti.** Diambil dari sedikit produk di katalog stockist.
- **"Bukan pelanggan"** baru dibandingkan dengan daftar stockist, belum dengan Salesforce.
- **Peringkat brand memakai bobot sama** dan belum diuji terhadap penilaian tim wholesale.
- **Harga** hanya membandingkan produk yang ditulis dalam USD.